# LAB 07 — Transformer Architecture

Notebook ghi lại kết quả được tạo bởi `run_lab7.py`. Chạy cell kế tiếp từ thư mục gốc repository để tái tạo toàn bộ thí nghiệm và hình.

In [ ]:
from pathlib import Path
import subprocess, sys
subprocess.run([sys.executable, str(Path('lab7/run_lab7.py'))], check=True)

## Transformer block tối giản

`TransformerBlock` tự ghép Q/K/V projections, scaled dot-product multi-head attention, output projection, residual + LayerNorm, GELU feed-forward và residual + LayerNorm. Không dùng `TransformerEncoderLayer`. Positional encoding được cộng vào token embeddings trước block.

In [ ]:
from lab7.positional_encoding import sinusoidal_positional_encoding
from lab7.transformer_block import TransformerBlock
import torch
block = TransformerBlock(128, 4, 256)
x = torch.randn(8, 100, 128)
y, attention = block(x, return_attention=True)
print('Q/K/V: (8, 4, 100, 32); scores:', tuple(attention.shape), 'output:', tuple(y.shape))

## Số liệu thí nghiệm lần chạy được lưu

Kết quả có thể khác theo môi trường và thời gian huấn luyện; xem `results.csv`, `experiment_details.json` và các hình đầu ra. Các phép so sánh giữ cố định split, vocabulary, embedding dimension, optimizer, batch size và epoch; ở thí nghiệm head chỉ thay số head, ở thí nghiệm độ dài chỉ thay max length.

In [ ]:
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv('lab7/results.csv'))
display(Image(filename='lab7/positional_encoding.png'))
display(Image(filename='lab7/attention_heads.png'))
display(Image(filename='lab7/learning_curves.png'))
display(Image(filename='lab7/confusion_matrix.png'))

### Kết quả mô hình chính

Bốn heads, độ dài 100, 2 epochs: accuracy **0.708**, precision **0.733**, recall **0.654**, F1 **0.691**. Đây là baseline học từ đầu trên tập con IMDb, không phải pretrained Transformer.

Application examples được lưu trong `experiment_details.json`: negative (0.205) — I really enjoyed this movie.; positive (0.860) — The film begins slowly, but the acting becomes powerful and the ending is wonderful.; negative (0.089) — I did not like this movie.; negative (0.388) — I expected a terrible film, but it was surprisingly good.; negative (0.062) — The movie was not bad, although it was not especially memorable.